# 03. Đánh giá từ bằng chứng thực nghiệm đã lưu
 Notebook cũ dùng KPI và độ trễ ghi sẵn. Bản này đọc dự đoán, tính lại metric bằng `src/evaluation/metrics.py`, kiểm tra khớp bản tổng hợp và hiển thị ca thất bại.

 Ba phạm vi được trình bày riêng: **benchmark offline cơ bản**, **hồi quy handler local**, và **Brave UI/hybrid**. Không gộp chúng thành một tỷ lệ.
 Notebook phân tích các lượt đã lưu, không tự chạy lại website hay tuyên bố mã hiện tại đã qua kiểm thử web. Đường dẫn được cố định ở ô cấu hình để tái lập, không tự chọn lượt có điểm cao nhất.

In [ ]:
from pathlib import Path
import sys
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/core').is_dir() and (p / 'notebooks').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the project root or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from notebooks._support import read_json, read_image, checked_crop, sha256
%matplotlib inline
plt.rcParams.update({'figure.figsize': (11, 4), 'font.size': 10})
print('Project:', ROOT)
print('Python:', sys.executable)

In [ ]:
from notebooks._support import load_benchmark, hash_status, regression_frame, compare_regressions
from src.evaluation.metrics import text_metrics

BENCHMARK_RUN = ROOT / 'data/benchmarks/20261105T084132_896570Z'
LOCAL_BEFORE = ROOT / 'data/recognition_regression/20261105T091906_946715Z_hardening_before_local'
LOCAL_AFTER = ROOT / 'data/recognition_regression/20261105T093925_904428Z_hardening_final_local'
HARDENING = ROOT / 'data/recognition_regression/hardening_20261105'
UI_BEFORE = HARDENING / 'before_ui/results.json'
UI_AFTER = HARDENING / 'after_ui/results.json'

report, predictions = load_benchmark(BENCHMARK_RUN)
manifest = report['manifest']
print('Benchmark đã lưu:', manifest['run_id'])
print('Phạm vi:', manifest['scope'])
print('Nguồn dữ liệu:', manifest['dataset_kind'])
print('Ảnh đã chấm / toàn bộ:', manifest['evaluated_images'], '/', manifest['dataset_images_total'])
print('metrics.json SHA256:', sha256(BENCHMARK_RUN / 'metrics.json'))
print('predictions.json SHA256:', sha256(BENCHMARK_RUN / 'predictions.json'))
print('Đã tính lại và đối chiếu toàn bộ metric tổng hợp / theo nhóm từ predictions.json.')
display(hash_status(manifest['source_sha256']))
print('changed = mã hiện tại khác mã của lượt lưu; kết quả này là lịch sử, không phải phép đo mã hiện tại.')
dataset_status = hash_status(manifest['image_sha256'], Path(manifest['dataset']))
display(dataset_status.groupby('status').size().rename('images').to_frame())
print('Không bao gồm trong latency local:', ', '.join(manifest['excluded_from_pipeline']))

## Benchmark offline: detection, OCR crop GT và end-to-end
 AP50 là AP một lớp ở IoU 0,5 trên proposal detector giữ lại; không phải `YOLO.val()`.
 Precision/recall vận hành dùng ngưỡng confidence của manifest. CER tính cả biển bỏ sót và chữ dư, có thể vượt 100%.

In [ ]:
overall = report['overall']
detection = overall['detection']
display(pd.DataFrame([{k:v for k,v in detection.items() if k != 'pr_curve'}]))
display(pd.DataFrame({key: overall[key] for key in
                      ['local_end_to_end', 'ground_truth_crop_ocr_raw', 'ground_truth_crop_ocr_formatted']}).T)
categories = pd.DataFrame([
    {'category': name, 'images': data['images'], 'detection_recall': data['detection']['recall'],
     'exact_image_rate': data['local_end_to_end']['exact_image_accuracy'],
     'cer': data['local_end_to_end']['cer']}
    for name, data in report['by_category'].items()]).set_index('category')
display(categories)
display(pd.DataFrame(overall['timing']).T)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
curve = pd.DataFrame(detection['pr_curve'])
if not curve.empty:
    axes[0].step(curve.recall, curve.precision, where='post')
axes[0].set(xlabel='Recall', ylabel='Precision', xlim=(0,1), ylim=(0,1), title='PR từ dự đoán đã lưu')
categories[['detection_recall', 'exact_image_rate']].plot.bar(ax=axes[1], ylim=(0,1), rot=20)
axes[1].set_title('Offline synthetic: kết quả theo nhóm')
plt.tight_layout()
plt.show()
failures = []
for record in predictions:
    if text_metrics([record])['exact_image_accuracy'] != 1:
        failures.append({'file': record['file'], 'category': record['category'],
                         'expected': [p['text'] for p in record['truth']],
                         'actual': [p['text'] for p in record['predictions']]})
print('Số ảnh thất bại trong benchmark:', len(failures))
display(pd.DataFrame(failures).head(12))

## Hồi quy local: so sánh đúng ảnh và đúng nhãn
 So sánh giao của hai manifest bằng ID, SHA256 ảnh, nhãn và loại ảnh. Dùng multiset giữ số lần lặp và cách chia prefix/serial, nên thiếu hoặc thêm biển đều bị tính sai.
 Các ca dùng để sửa lỗi là dữ liệu phát triển, không phải tập kiểm thử độc lập. Những ảnh thêm ở lượt sau được thống kê riêng.

In [ ]:
before_report, before = regression_frame(LOCAL_BEFORE / 'results.json', LOCAL_BEFORE / 'cases.json')
after_report, after = regression_frame(LOCAL_AFTER / 'results.json', LOCAL_AFTER / 'cases.json')
paired = compare_regressions(before_report, before, after_report, after)
print('Giao hai tập cố định:', len(paired), 'ảnh')
print('Trước:', int(paired.exact_before.sum()), '/', len(paired),
      '; Sau:', int(paired.exact_after.sum()), '/', len(paired))
print('Ca chỉ có ở lượt sau:', len(after) - len(paired))
display(paired[['id', 'expected_before', 'actual_before', 'actual_after', 'exact_before', 'exact_after']])
display(after.groupby('kind').agg(images=('id','size'), exact=('exact','sum'), rate=('exact','mean')))
print('Toàn bộ lượt local sau:', int(after.exact.sum()), '/', len(after))
print('Cloud attempts:', after.cloud_attempts.sum())
display(hash_status(after_report['model_hashes'], ROOT / 'models'))
display(hash_status(after_report['code_hashes']).query("status != 'same'"))
remaining = after.loc[~after.exact]
display(remaining[['id', 'kind', 'expected', 'actual', 'wall_ms']])

In [ ]:
case_paths = {c['id']: c['path'] for c in read_json(LOCAL_AFTER / 'cases.json')['cases']}
if not remaining.empty:
    fig, axes = plt.subplots(1, len(remaining), figsize=(12, 5), squeeze=False)
    for ax, (_, case) in zip(axes.flat, remaining.iterrows()):
        ax.imshow(cv2.cvtColor(read_image(case_paths[case['id']]), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{case['id']}\nGT: {', '.join(case['expected'])}\nOCR: {case['actual']}", fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Không có ca sai trong lượt local đã chọn; không suy rộng ra ảnh chưa kiểm thử.')

## Kiểm thử qua Brave: bằng chứng riêng
 HTTP, cookie, frontend và cloud làm phạm vi này khác gọi handler local. Chỉ trình bày lượt có file chứng cứ; không lấy kết quả local thay cho kiểm thử web.

In [ ]:
ui_before_report = None
if UI_BEFORE.is_file():
    ui_before_report, ui_before = regression_frame(UI_BEFORE, HARDENING / 'cases.json')
    print('Brave trước sửa:', int(ui_before.exact.sum()), '/', len(ui_before))
    display(ui_before.loc[~ui_before.exact, ['id', 'expected', 'actual', 'cloud_attempts']])
else:
    print('CHƯA CÓ bằng chứng Brave trước sửa.')
if UI_AFTER.is_file():
    ui_after_report, ui_after = regression_frame(UI_AFTER, HARDENING / 'cases.json')
    print('Brave sau sửa:', int(ui_after.exact.sum()), '/', len(ui_after))
    if ui_before_report is not None:
        ui_paired = compare_regressions(ui_before_report, ui_before, ui_after_report, ui_after)
        display(ui_paired[['id', 'exact_before', 'exact_after']])
else:
    print('CHƯA XÁC MINH BRAVE SAU SỬA: chưa có after_ui/results.json.')
print('Các kết quả trên là phép đo đã lưu; không có lần chạy web mới trong notebook này.')